# 05 — Data Modeling & Business Metrics

Build a small **star schema** and a reusable reporting table using the Silver and Gold Delta tables produced in `04_Medallion_Architecture`.

- **Input:** `workspace.silver.sales`, `stores`, `transactions`, and `workspace.gold.store_holiday_flags`.
- **Output:** three dimensions, two facts, and one reporting mart in `workspace.gold`.
- **Grain:** `fact_sales` = store × date × product family; `fact_transactions` = store × date.

This notebook **does not reread CSVs or rebuild Bronze/Silver**. It overwrites only its six named Gold tables. Sales values are **quantities, not revenue**. A missing transaction row must not be treated as zero transactions.

## 1. Setup and input tables
Reuse the Silver tables created in 04. Check the required source tables before building the model.

In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.silver"
GOLD = "workspace.gold"

silver_sales = spark.table(f"{SILVER}.sales")
silver_stores = spark.table(f"{SILVER}.stores")
silver_transactions = spark.table(f"{SILVER}.transactions")
holiday_flags = spark.table(f"{GOLD}.store_holiday_flags")

def save_gold(df, name):
    target = f"{GOLD}.{name}"
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    print("Saved:", target)

print("Input tables loaded from Silver and Gold.")

## 2. Dimensions
Dimensions describe **when**, **where**, and **which product family**. We use the existing natural business keys (`date`, `store_nbr`, `family`) rather than introducing unnecessary surrogate keys for this small model.

The date dimension includes **every calendar date**, even the four Dec 25 dates absent from historical sales. Missing fact records are **not** interpreted as zero sales.

In [0]:
# Calendar dimension covers every day within the historical sales date range.
date_bounds = silver_sales.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date")
)

dim_date = (
    date_bounds
    .select(F.explode(F.sequence("start_date", "end_date")).alias("date"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_start", F.trunc("date", "month"))
    .withColumn("day_of_week", F.dayofweek("date"))  # Sunday = 1
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7))
)

dim_store = silver_stores.select(
    "store_nbr", "city", "state", "type", "cluster"
)

dim_product_family = (
    silver_sales.select("family")
    .distinct()
)

save_gold(dim_date, "dim_date")
save_gold(dim_store, "dim_store")
save_gold(dim_product_family, "dim_product_family")

## 3. Fact tables
Keep the two fact tables **separate**: one record per store/date/family for sales, and one per store/date for transactions. Joining transactions directly to category-level sales would repeat the transaction count across families.

In [0]:
fact_sales = silver_sales.select(
    "id", "date", "store_nbr", "family",
    F.col("sales").alias("sales_qty"),
    F.col("onpromotion").alias("promoted_item_count")
)

fact_transactions = silver_transactions.select(
    "date", "store_nbr",
    F.col("transactions").alias("transaction_count")
)

save_gold(fact_sales, "fact_sales")
save_gold(fact_transactions, "fact_transactions")

## 4. Monthly beverage reporting mart
Create a Power BI–friendly table at the **store × month** grain.

First aggregate beverage sales and transactions **separately**, then join the monthly results. This avoids multiplying transactions across product categories. Holiday counts refer only to dates with beverage sales records.

In [0]:
# Each holiday flag has a unique (date, store_nbr) key, validated in 04.
date_lookup = dim_date.select("date", "year", "month")

monthly_beverages = (
    fact_sales
    .filter(F.col("family") == "BEVERAGES")
    .join(date_lookup, "date", "inner")
    .join(holiday_flags, ["date", "store_nbr"], "left")
    .groupBy("year", "month", "store_nbr")
    .agg(
        F.sum("sales_qty").alias("total_beverage_sales_qty"),
        F.countDistinct("date").alias("sales_record_days"),
        F.sum(F.when(F.col("promoted_item_count") > 0, 1).otherwise(0)).alias("promotion_days"),
        F.sum(F.when(F.col("is_holiday") == True, 1).otherwise(0)).alias("holiday_sales_days")
    )
    .withColumn(
        "avg_daily_beverage_sales_qty",
        F.round(F.col("total_beverage_sales_qty") / F.col("sales_record_days"), 2)
    )
)

monthly_transactions = (
    fact_transactions
    .join(date_lookup, "date", "inner")
    .groupBy("year", "month", "store_nbr")
    .agg(
        F.sum("transaction_count").alias("total_recorded_transactions"),
        F.count("transaction_count").alias("transaction_record_days")
    )
    .withColumn(
        "avg_transactions_per_recorded_day",
        F.round(
            F.col("total_recorded_transactions") / F.col("transaction_record_days"), 2
        )
    )
)

monthly_beverage_kpis = (
    monthly_beverages
    .join(monthly_transactions, ["year", "month", "store_nbr"], "left")
    .join(dim_store.select("store_nbr", "city", "state", "type"), "store_nbr", "left")
    .withColumn("month_start", F.make_date("year", "month", F.lit(1)))
    .select(
        "month_start", "year", "month", "store_nbr", "city", "state", "type",
        "total_beverage_sales_qty", "sales_record_days", "avg_daily_beverage_sales_qty",
        "promotion_days", "holiday_sales_days",
        "total_recorded_transactions", "transaction_record_days",
        "avg_transactions_per_recorded_day"
    )
)

save_gold(monthly_beverage_kpis, "mart_monthly_beverage_kpis")

## 5. Validation
Check business-key uniqueness, row counts, dimension references, and the mart's aggregated beverage total. If any assertion fails, fix the model before using it for reporting.

In [0]:
def assert_unique(df, keys, name):
    has_duplicates = (
        df.groupBy(*keys).count()
        .filter(F.col("count") > 1).limit(1).count()
    )
    assert has_duplicates == 0, f"Duplicate keys in {name}: {keys}"
    print("Unique keys:", name)

for name, df, keys in [
    ("dim_date", dim_date, ["date"]),
    ("dim_store", dim_store, ["store_nbr"]),
    ("dim_product_family", dim_product_family, ["family"]),
    ("fact_sales", fact_sales, ["date", "store_nbr", "family"]),
    ("fact_transactions", fact_transactions, ["date", "store_nbr"]),
    ("mart_monthly_beverage_kpis", monthly_beverage_kpis, ["year", "month", "store_nbr"])
]:
    assert_unique(df, keys, name)

assert fact_sales.count() == silver_sales.count(), "Sales records were lost"
assert fact_transactions.count() == silver_transactions.count(), "Transaction records were lost"

# Ensure every fact key exists in its matching dimension.
for df, source_key, dim, dim_key, description in [
    (fact_sales, "date", dim_date, "date", "sales date"),
    (fact_sales, "store_nbr", dim_store, "store_nbr", "sales store"),
    (fact_sales, "family", dim_product_family, "family", "product family"),
    (fact_transactions, "date", dim_date, "date", "transaction date"),
    (fact_transactions, "store_nbr", dim_store, "store_nbr", "transaction store")
]:
    unknown = (
        df.select(source_key).distinct()
        .join(dim.select(F.col(dim_key).alias(source_key)), source_key, "left_anti")
        .limit(1).count()
    )
    assert unknown == 0, f"Unknown dimension key: {description}"

# Compare total beverage quantities before/after monthly aggregation.
source_total = (
    fact_sales.filter(F.col("family") == "BEVERAGES")
    .agg(F.sum("sales_qty").alias("qty")).first()["qty"]
)
mart_total = monthly_beverage_kpis.agg(
    F.sum("total_beverage_sales_qty").alias("qty")
).first()["qty"]
assert abs(source_total - mart_total) < 0.01, "Mart beverage total differs from source"

for name in [
    "dim_date", "dim_store", "dim_product_family", "fact_sales",
    "fact_transactions", "mart_monthly_beverage_kpis"
]:
    print(f"{GOLD}.{name}: {spark.table(f'{GOLD}.{name}').count():,} rows")

print("Validation passed.")
display(spark.table(f"{GOLD}.mart_monthly_beverage_kpis").orderBy("month_start", "store_nbr").limit(10))

## 6. How to use these tables

**Star schema**

- `fact_sales` → `dim_date`, `dim_store`, `dim_product_family`.
- `fact_transactions` → `dim_date`, `dim_store`.

**For Power BI:** start with `workspace.gold.mart_monthly_beverage_kpis`. It contains one record per store and month, with pre-aggregated beverage and transaction measures.

**Interpretation:** `total_beverage_sales_qty` is not revenue. `total_recorded_transactions` covers only days with available transaction records; NULL is not zero. This is a full-refresh batch model; scheduling, incremental updates, and BI dashboard creation are outside this Notebook.